In [1]:
import torch
import pickle
import argparse
from tqdm import tqdm
import numpy as np
import pandas as pd
import os
from torch.nn.utils.rnn import pad_sequence
from terrarium.models import get_model, load_model
from terrarium.dataloaders import get_dataloader

Terrarium.tokenizers could not be imported, likely not yet developed.
Terrarium.train could not be imported, likely not yet developed.


In [2]:
model_name = 'gpt2_v1'
ckpt_path = 'out/gpt2_v1/7M/ckpt.pt'
data_dir = 'data/v_32768'
dl_name = 'standard_inference'
mc_samples = 2
batch_size = 1
max_seq_len = 512
total_sequences = 1
seed_lengths = list(range(0, 501, 50))
data_prefix = 'val'
summary_only = False
out_dir = './'
device = 'cpu'

In [3]:
model = load_model(model_name,ckpt_path,device_override='cpu').to(device)
checkpoint = torch.load(ckpt_path, map_location='cpu', weights_only=False) # always first to cpu

In [4]:
with open(f"{data_dir}/tok.pkl",'rb') as f:
    tok = pickle.load(f)
sos_id = tok.encode('<|sos|>')
dl = get_dataloader(dl_name,batch_size,max_seq_len,f"{data_dir}/{data_prefix}_token_ids.bin",sos_id,device)

In [5]:
# check if dataset has enough sequences
if len(dl.split_locs)-1 < total_sequences:
    total_sequences = len(dl.split_locs)-1
    print(f'Warning: requested total_sequences reduced to {total_sequences} due to data size.')
n_batches = total_sequences // batch_size

In [8]:
# create the holder for results
results = {
    s: {
        'true_token': [],
        'sampled_token': [],
        'top1_token': [],
        'cross_entropy_loss': []
    }
    for s in seed_lengths
}

In [ ]:
def mc_sampling(model,b,max_seq_len,n_samples,search_ids):
    
    # returns mc sampled ratings from b
    with torch.no_grad(), torch.autocast(device_type=device, dtype=torch.bfloat16):
        output = model._generate(b,max_seq_len-b.shape[1]) 

    search_ids = torch.as_tensor(search_ids, device=output.device)
    mask = torch.isin(output, search_ids)   # (n_samples, seq_len)

    # first occurrence index per sample
    first_occ = mask.float().argmax(dim=1)

    # mark samples with no occurrence
    no_match = ~mask.any(dim=1)
    first_occ[no_match] = -1

    # guard: if first token is a search id (or invalid index)
    first_occ[first_occ <= 0] = -1

    # extract attribute ids
    attr_ids = output[torch.arange(n_samples), first_occ]

    mc_attr_ids = []
    n_errors = 0
    for r,f in zip(attr_ids.tolist(),first_occ.tolist()):
        if r not in search_ids or f == 0:
            n_errors+=1
        else:
            idx = (search_ids == r).nonzero(as_tuple=True)[0].item()
            mc_attr_ids.append(idx)

    mc_attr_ids = np.bincount(mc_attr_ids, minlength=len(search_ids))
    return mc_attr_ids

In [7]:
# --- run the inference over batches --- #
for b in tqdm(range(n_batches)):
    x,y = dl.next_batch_mc(n_samples=mc_samples)
    tokens = torch.tensor([seq[-1].item() for seq in y], dtype=torch.long, device=device)

    # assemble x and y from the return sequences t.
    x_pad = torch.zeros((len(x), max_seq_len), dtype=torch.long)
    y_pad = torch.zeros((len(y), max_seq_len), dtype=torch.long)
    seq_lengths = torch.zeros((len(x),), dtype=torch.long)
    for i, (xi, yi) in enumerate(zip(x, y)):
        L = len(xi)
        seq_lengths[i] = L
        x_pad[i, :L] = xi
        y_pad[i, :L] = yi
    x_pad,y_pad,seq_lengths = x_pad.to(device),y_pad.to(device),seq_lengths.to(device)

    for 

    
    # generate out to max_seq_len
    gen = mc_sampling(model,x[:, :s],max_seq_len,mc_samples,search_ids)
    B,T,V = logits.shape
    token_probs = torch.softmax(logits,dim=-1)

    # for the observed next token, get cond prob, sampled outcome, top1 outcome, nll
    sampled_token = torch.multinomial(token_probs.view(-1,V),num_samples=1).view(B,T) # B,T
    top1_token = torch.argmax(token_probs,dim=-1) # B,T
    nll = -torch.log(torch.gather(token_probs,2,y_pad.unsqueeze(-1)).clamp_min(1e-9)).squeeze(-1) # B,T

    rows = torch.arange(B, device=device)
    last_idx = (seq_lengths - 1).clamp_min(0)  # B,

    # if has 
    
    last_sampled = sampled_token[rows, last_idx] # (B,)
    last_top1 = top1_token[rows, last_idx] # (B,)
    last_ce = nll[rows, last_idx] # (B,)

    for s in seed_lengths:
        rows_to_use = np.where(seq_lengths > s)[0]
        if len(rows_to_use) == 0:
            continue
        results[s]['true_token'].append(attrs[rows_to_use].tolist())
        results[s]['sampled_token'].append(sampled_outcome[rows_to_use, s-2].tolist())
        results[s]['top1_token'].append(top1_outcome[rows_to_use, s-2].tolist())
        results[s]['cross_entropy_loss'].append(nll[rows_to_use, s-2, attrs[rows_to_use]].tolist())

  0%|          | 0/1 [00:00<?, ?it/s]


NameError: name 'cond_prob_obs' is not defined

In [51]:
# ---- data post processing ---- #
for s in seed_lengths:
    results[s]['true_attrs'] = [item for sublist in results[s]['true_attrs'] for item in sublist]
    results[s]['sampled_outcome'] = [item for sublist in results[s]['sampled_outcome'] for item in sublist]
    results[s]['top1_outcome'] = [item for sublist in results[s]['top1_outcome'] for item in sublist]
    results[s]['cross_entropy_loss'] = [item for sublist in results[s]['cross_entropy_loss'] for item in sublist]

In [52]:
# now to df for csv output, making seed length a col so can contatenate all into one file
all_dfs = []
for s in seed_lengths:
    df = pd.DataFrame({
        'seed_length': [s]*len(results[s]['true_attrs']),
        'true_attrs': results[s]['true_attrs'],
        'sampled_outcome': results[s]['sampled_outcome'],
        'top1_outcome': results[s]['top1_outcome'],
        'cross_entropy_loss': results[s]['cross_entropy_loss']
    })
    all_dfs.append(df)

In [54]:
# concatenate all dataframes
final_df = pd.concat(all_dfs, ignore_index=True)
# final_df.to_csv('./out/online_classification_results.csv', index=False) # not sure want to save, could be massive
if not summary_only:
    final_df.to_csv(os.path.join(out_dir,'online_classification_results.csv'), index=False)

In [55]:
# overall summary by seed length
df_overall_summary = final_df.groupby('seed_length').agg(
    n_attrs=('true_attrs', 'count'),
    sampled_accuracy=('sampled_outcome', lambda x: np.mean(x == final_df.loc[x.index, 'true_attrs'])),
    top1_accuracy=('top1_outcome', lambda x: np.mean(x == final_df.loc[x.index, 'true_attrs'])),
    cross_entropy_mean=('cross_entropy_loss', 'mean')
)

In [56]:
# summary by seed and true rating
df_stratified_summary = (
    final_df
    .groupby(['seed_length', 'true_attrs'])
    .agg(
        n_attrs=('true_attrs', 'count'),
        sampled_accuracy=('sampled_outcome', lambda x: np.mean(x == final_df.loc[x.index, 'true_attrs'])),
        top1_accuracy=('top1_outcome', lambda x: np.mean(x == final_df.loc[x.index, 'true_attrs'])),
        cross_entropy_mean=('cross_entropy_loss', 'mean')
    )
    .reset_index()  # make seed_length, true_attrs into columns again
)

In [57]:
df_overall_summary.to_csv(os.path.join(out_dir,'online_classification_overall_summary.csv'),index=False)
df_stratified_summary.to_csv(os.path.join(out_dir,'online_classification_stratified_summary.csv'),index=False)